# Лабораторная работа: Продвинутый NumPy

**ФИО:** Парамонов Даниил Дмитриевич  
**Группа:** 1521741, 3 курс

Темы: числа с плавающей точкой, индексация и broadcasting, memory-mapped файлы и кэш, разреженные матрицы, Cython.

In [1]:
import sys, os
import numpy as np
import scipy
import scipy.sparse as sp
print('Python', sys.version.split()[0], '| NumPy', np.__version__, '| SciPy', scipy.__version__)

Python 3.13.18 | NumPy 2.5.3 | SciPy 1.18.1


## Задача 1. Числа с плавающей точкой

### 1.1 Сумма `float32`-массива `[0.1, 0.2, 0.3]` и сравнение с точным значением

In [2]:
from decimal import Decimal

a32 = np.array([0.1, 0.2, 0.3], dtype=np.float32)

# Какие значения на самом деле хранятся в float32 (Decimal(float) даёт точное десятичное значение)
for x, name in zip(a32, ['0.1', '0.2', '0.3']):
    print(f'{name} хранится как {Decimal(float(x))}')

s32 = a32.sum()
exact = Decimal('0.6')
err = abs(Decimal(float(s32)) - exact)
print()
print('dtype суммы:        ', s32.dtype)
print('Сумма float32:      ', Decimal(float(s32)))
print('Точное значение:    ', exact)
print('Абсолютная ошибка:  ', err)
print('Относительная ошибка:', err / exact)

# Для сравнения: float64
a64 = np.array([0.1, 0.2, 0.3], dtype=np.float64)
s64 = a64.sum()
err64 = abs(Decimal(float(s64)) - exact)
print()
print('Сумма float64:      ', Decimal(float(s64)))
print('Абсолютная ошибка:  ', err64)
print('0.1 + 0.2 == 0.3 (float64)?', np.float64(0.1) + np.float64(0.2) == np.float64(0.3))

0.1 хранится как 0.100000001490116119384765625
0.2 хранится как 0.20000000298023223876953125
0.3 хранится как 0.300000011920928955078125

dtype суммы:         float32
Сумма float32:       0.60000002384185791015625
Точное значение:     0.6
Абсолютная ошибка:   2.384185791015625E-8
Относительная ошибка: 3.973642985026041666666666667E-8

Сумма float64:       0.600000000000000088817841970012523233890533447265625
Абсолютная ошибка:   8.881784197001252323389053345E-17
0.1 + 0.2 == 0.3 (float64)? False


**Вывод 1.1.** Числа 0.1, 0.2, 0.3 не имеют конечного двоичного представления, поэтому уже при записи в массив они округляются, и сумма не равна точному значению 0.6. У `float32` ошибка порядка 1e-8, у `float64` - порядка 1e-17. Поэтому `==` для чисел с плавающей точкой использовать нельзя, нужно `np.isclose` / `np.allclose`.

### 1.2 Машинное эпсилон через `np.finfo`

In [3]:
for t in (np.float32, np.float64):
    fi = np.finfo(t)
    one = t(1)
    print(f'{t.__name__}: eps = {fi.eps}, бит мантиссы = {fi.nmant}, десятичных знаков ~ {fi.precision}')
    print(f'    1 + eps   > 1 : {one + fi.eps > one}')
    print(f'    1 + eps/2 == 1: {one + fi.eps / t(2) == one}')

# Проверка: находим эпсилон вручную делением пополам
def my_eps(t):
    e = t(1)
    while t(1) + e / t(2) > t(1):
        e = e / t(2)
    return e

print()
print('Ручной поиск float32:', my_eps(np.float32), '| np.finfo:', np.finfo(np.float32).eps)
print('Ручной поиск float64:', my_eps(np.float64), '| np.finfo:', np.finfo(np.float64).eps)

float32: eps = 1.1920928955078125e-07, бит мантиссы = 23, десятичных знаков ~ 6
    1 + eps   > 1 : True
    1 + eps/2 == 1: True
float64: eps = 2.220446049250313e-16, бит мантиссы = 52, десятичных знаков ~ 15
    1 + eps   > 1 : True
    1 + eps/2 == 1: True

Ручной поиск float32: 1.1920929e-07 | np.finfo: 1.1920929e-07
Ручной поиск float64: 2.220446049250313e-16 | np.finfo: 2.220446049250313e-16


**Вывод 1.2.** Машинное эпсилон - наименьшее `eps`, для которого `1 + eps > 1`. Для `float32` оно равно 2⁻²³ ≈ 1.19·10⁻⁷ (≈6-7 десятичных знаков), для `float64` - 2⁻⁵² ≈ 2.22·10⁻¹⁶ (≈15-16 знаков). Это граница относительной точности типа.

### 1.3 Катастрофическая отмена при вычитании близких чисел

In [4]:
# Пример 1: (1 + x) - 1 при x меньше эпсилон float32
print('Пример 1: (1 + x) - 1, где x = 1e-8')
for t in (np.float32, np.float64):
    x = t(1e-8)
    r = (t(1) + x) - t(1)
    print(f'  {t.__name__}: результат = {r!r}  (ожидалось 1e-08)')

# Пример 2: sqrt(x+1) - sqrt(x) при больших x
print()
print('Пример 2: sqrt(x+1) - sqrt(x), x = 1e12 (float64)')
x = np.float64(1e12)
naive  = np.sqrt(x + 1) - np.sqrt(x)           # вычитание близких чисел
stable = 1.0 / (np.sqrt(x + 1) + np.sqrt(x))   # алгебраически то же самое, без вычитания
print('  наивная формула :', naive)
print('  устойчивая форма:', stable)
print('  относительная ошибка наивной формулы:', abs(naive - stable) / stable)

# Пример 3: то же в float32
x32 = np.float32(1e6)
naive32  = np.sqrt(x32 + np.float32(1)) - np.sqrt(x32)
stable32 = np.float32(1) / (np.sqrt(x32 + np.float32(1)) + np.sqrt(x32))
print()
print('Пример 3: x = 1e6 (float32)')
print('  наивная формула :', naive32)
print('  устойчивая форма:', stable32)
print('  относительная ошибка наивной формулы:', abs(naive32 - stable32) / stable32)

Пример 1: (1 + x) - 1, где x = 1e-8
  float32: результат = np.float32(0.0)  (ожидалось 1e-08)
  float64: результат = np.float64(9.99999993922529e-09)  (ожидалось 1e-08)

Пример 2: sqrt(x+1) - sqrt(x), x = 1e12 (float64)
  наивная формула : 5.00003807246685e-07
  устойчивая форма: 4.999999999998749e-07
  относительная ошибка наивной формулы: 7.614493620189787e-06

Пример 3: x = 1e6 (float32)
  наивная формула : 0.00048828125
  устойчивая форма: 0.00049999985
  относительная ошибка наивной формулы: 0.023437206


**Вывод 1.3.** При вычитании почти равных чисел старшие верные цифры сокращаются, и остаются в основном накопленные ошибки округления. Относительная ошибка резко растёт: в `float32` слагаемое `1e-8` теряется полностью (результат 0), а в примере с корнями потеряна заметная часть значащих цифр. Лечится алгебраическим преобразованием выражения (как в «устойчивой форме») или повышением точности.

## Задача 2. Индексация, срезы и broadcasting

### 2.1 Базовые срезы

In [5]:
# Массив (4, 5) со значениями от -10 до 9, чтобы были отрицательные элементы (нужно для 2.3)
A = np.arange(-10, 10).reshape(4, 5)
print('A =')
print(A)

print('\nВторая строка (индекс 1):        ', A[1])
print('Столбец с индексом 3:            ', A[:, 3])
print('Подмассив строки 1-3, столбцы 2-4:')
print(A[1:4, 2:5])

A =
[[-10  -9  -8  -7  -6]
 [ -5  -4  -3  -2  -1]
 [  0   1   2   3   4]
 [  5   6   7   8   9]]

Вторая строка (индекс 1):         [-5 -4 -3 -2 -1]
Столбец с индексом 3:             [-7 -2  3  8]
Подмассив строки 1-3, столбцы 2-4:
[[-3 -2 -1]
 [ 2  3  4]
 [ 7  8  9]]


Строки 1-3 и столбцы 2-4 понимаются как **индексы** (включительно), поэтому срез `A[1:4, 2:5]`: правая граница среза в Python не включается. Срезы возвращают *view* (представление) исходного массива, а не копию.

### 2.2 Fancy-индексация

In [6]:
rows = [0, 2, 3]
cols = [1, 4]

# Прямая запись A[rows, cols] выбирает ПАРЫ (0,1), (2,4), (3,?) и требует одинаковой длины списков
try:
    A[rows, cols]
except IndexError as e:
    print('A[rows, cols] -', type(e).__name__ + ':', e)

# Нужны все комбинации строк и столбцов (декартово произведение) - используем np.ix_
sub = A[np.ix_(rows, cols)]
print('\nA[np.ix_([0, 2, 3], [1, 4])] =')
print(sub, ' форма:', sub.shape)

# Эквивалентная запись через broadcasting индексов
print('\nЭквивалентно A[[[0],[2],[3]], [1, 4]]:')
print(A[[[0], [2], [3]], [1, 4]])

# Fancy-индексация возвращает КОПИЮ
sub[0, 0] = 999
print('\nA[0, 1] после изменения копии:', A[0, 1], '(исходный массив не изменился)')

A[rows, cols] - IndexError: shape mismatch: indexing arrays could not be broadcast together with shapes (3,) (2,) 

A[np.ix_([0, 2, 3], [1, 4])] =
[[-9 -6]
 [ 1  4]
 [ 6  9]]  форма: (3, 2)

Эквивалентно A[[[0],[2],[3]], [1, 4]]:
[[-9 -6]
 [ 1  4]
 [ 6  9]]

A[0, 1] после изменения копии: -9 (исходный массив не изменился)


**Вывод 2.2.** Запись `A[[0,2,3], [1,4]]` в NumPy означает поэлементное соединение индексов в пары, и при разной длине списков вызывает ошибку. Чтобы взять элементы на пересечении выбранных строк и столбцов, используют `np.ix_` (или индексы нужной формы для broadcasting). Результат fancy-индексации - копия.

### 2.3 Логическая индексация: замена отрицательных элементов на 0

In [7]:
B_ = A.copy()
mask = B_ < 0
print('Маска (True там, где элемент отрицательный):')
print(mask)
B_[mask] = 0          # то же, что B_[B_ < 0] = 0
print('\nПосле замены:')
print(B_)
print('Отрицательных элементов осталось:', (B_ < 0).sum())

Маска (True там, где элемент отрицательный):
[[ True  True  True  True  True]
 [ True  True  True  True  True]
 [False False False False False]
 [False False False False False]]

После замены:
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 1 2 3 4]
 [5 6 7 8 9]]
Отрицательных элементов осталось: 0


### 2.4 Broadcasting: массивы `(4, 1)` и `(1, 5)`

In [8]:
Bv = np.arange(1, 5).reshape(4, 1)      # форма (4, 1)
Cv = np.arange(10, 60, 10).reshape(1, 5)  # форма (1, 5)
R = Bv + Cv                              # без циклов
print('B =', Bv.ravel(), 'форма', Bv.shape)
print('C =', Cv.ravel(), 'форма', Cv.shape)
print('B + C, форма', R.shape)
print(R)

# Проверка: то же самое с помощью циклов
R_loop = np.empty((4, 5), dtype=int)
for i in range(4):
    for j in range(5):
        R_loop[i, j] = Bv[i, 0] + Cv[0, j]
print('\nСовпадает с результатом циклов:', np.array_equal(R, R_loop))

B = [1 2 3 4] форма (4, 1)
C = [10 20 30 40 50] форма (1, 5)
B + C, форма (4, 5)
[[11 21 31 41 51]
 [12 22 32 42 52]
 [13 23 33 43 53]
 [14 24 34 44 54]]

Совпадает с результатом циклов: True


**Вывод 2.4.** Правила broadcasting: формы сравниваются справа налево, размерности совместимы, если они равны или одна из них равна 1. Здесь `(4, 1)` и `(1, 5)` - `(4, 5)`: столбец «растягивается» вдоль столбцов, строка - вдоль строк. Копии данных при этом не создаются.

## Задача 3. Memory-mapped файлы и оптимизация кэша

### 3.1 Создание memory-mapped файла (1000, 1000), `float64`

In [9]:
fname = 'big_array.dat'
shape = (1000, 1000)

mm = np.memmap(fname, dtype=np.float64, mode='w+', shape=shape)
rng = np.random.default_rng(42)
mm[:] = rng.random(shape)     # заполняем случайными числами
mm.flush()                    # сбрасываем изменения на диск
print('Тип объекта      :', type(mm).__name__)
print('Размер файла, байт:', os.path.getsize(fname), '(ожидалось', shape[0] * shape[1] * 8, ')')
del mm                        # закрываем отображение

Тип объекта      : memmap
Размер файла, байт: 8000000 (ожидалось 8000000 )


### 3.2 Чтение отдельной строки и столбца без загрузки всего массива

In [10]:
mm = np.memmap(fname, dtype=np.float64, mode='r', shape=shape)   # только чтение

row = np.array(mm[3, :])   # np.array копирует в ОЗУ только эти 1000 элементов
col = np.array(mm[:, 3])
print('Строка 3 :', row.shape, row[:4], '...')
print('Столбец 3:', col.shape, col[:4], '...')

# Проверка: читаем ту же строку напрямую из файла
row_check = np.fromfile(fname, dtype=np.float64, count=1000, offset=3 * 1000 * 8)
print('Строка совпадает с прямым чтением файла:', np.array_equal(row, row_check))

# Строка лежит в файле подряд, а элементы столбца разбросаны с шагом в 8000 байт
print('\nШаг по строке  (байт):', mm.strides[1])
print('Шаг по столбцу (байт):', mm.strides[0])

Строка 3 : (1000,) [0.82922114 0.59409599 0.24485475 0.745675  ] ...
Столбец 3: (1000,) [0.69736803 0.15232671 0.65079459 0.745675  ] ...
Строка совпадает с прямым чтением файла: True

Шаг по строке  (байт): 8
Шаг по столбцу (байт): 8000


**Вывод 3.2.** `np.memmap` отображает файл в виртуальную память, и ОС подгружает только те страницы, к которым обратились. Срез `mm[3, :]` - один непрерывный фрагмент файла (8000 байт), а `mm[:, 3]` - 1000 отдельных элементов с шагом 8000 байт, то есть затрагивает ~1000 разных мест файла. Всё равно весь массив в ОЗУ не загружается.

### 3.3 Обход по строкам и по столбцам (`%timeit`)

In [11]:
a_c = np.array(mm)               # копия в ОЗУ в C-порядке (row-major)
a_f = np.asfortranarray(a_c)     # то же содержимое в Fortran-порядке (column-major)
print('a_c: C_CONTIGUOUS =', a_c.flags['C_CONTIGUOUS'], '| a_f: F_CONTIGUOUS =', a_f.flags['F_CONTIGUOUS'])

def sum_by_rows(a):
    s = 0.0
    for i in range(a.shape[0]):
        s += a[i, :].sum()
    return s

def sum_by_cols(a):
    s = 0.0
    for j in range(a.shape[1]):
        s += a[:, j].sum()
    return s

print('Суммы совпадают:', np.isclose(sum_by_rows(a_c), sum_by_cols(a_c)))

a_c: C_CONTIGUOUS = True | a_f: F_CONTIGUOUS = True
Суммы совпадают: True


In [12]:
print('C-порядок, обход по строкам:')
%timeit sum_by_rows(a_c)

C-порядок, обход по строкам:


1.21 ms ± 35.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [13]:
print('C-порядок, обход по столбцам:')
%timeit sum_by_cols(a_c)

C-порядок, обход по столбцам:


1.72 ms ± 76.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
print('F-порядок, обход по строкам:')
%timeit sum_by_rows(a_f)

F-порядок, обход по строкам:


1.73 ms ± 58.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
print('F-порядок, обход по столбцам:')
%timeit sum_by_cols(a_f)

F-порядок, обход по столбцам:


1.16 ms ± 50.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [16]:
print('memmap, обход по строкам:')
%timeit sum_by_rows(mm)

memmap, обход по строкам:


3.21 ms ± 166 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [17]:
print('memmap, обход по столбцам:')
%timeit sum_by_cols(mm)

memmap, обход по столбцам:


4.24 ms ± 166 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Вывод 3.3.** Результаты `%timeit` (1000×1000, `float64`):

| Массив | По строкам | По столбцам |
|---|---|---|
| C-порядок (row-major) в ОЗУ | ≈ 1.21 мс | ≈ 1.72 мс |
| Fortran-порядок (column-major) в ОЗУ | ≈ 1.73 мс | ≈ 1.16 мс |
| `np.memmap` (C-порядок) | ≈ 3.21 мс | ≈ 4.24 мс |

Быстрее обход в том порядке, в котором данные лежат в памяти: для C-массива это строки, для Fortran-массива - столбцы (результаты зеркальны). Процессор читает память кэш-линиями по 64 байта (8 значений `float64`). При обходе вдоль строки C-массива элементы идут подряд: одна загруженная линия даёт 8 полезных значений, а аппаратная предвыборка угадывает следующие. При обходе столбца соседние элементы отстоят на 8000 байт, каждый попадает в отдельную кэш-линию, из которой используется 1/8, растёт число промахов кэша. В нашем тесте разница умеренная (≈1.4 раза), потому что массив размером 8 МБ в значительной степени помещается в кэш процессора, а во времени присутствует и накладной расход Python-цикла; для массивов, заметно больших кэша, разрыв обычно увеличивается. У `memmap` время больше из-за доступа через отображение страниц (данные уже в страничном кэше ОС, поэтому реальных чтений с диска нет), но порядок обхода влияет так же.

## Задача 4. Разреженные матрицы

### 4.1 Матрица (5, 5), плотность ~20%, формат CSR

In [18]:
A_csr = sp.random(5, 5, density=0.2, format='csr', rng=42)
print('Формат:', A_csr.format, '| форма:', A_csr.shape)
print('Ненулевых элементов:', A_csr.nnz, 'из', 5 * 5, '- плотность', A_csr.nnz / 25)
print('\nПлотный вид:')
print(A_csr.toarray().round(3))
print('\nCSR хранит три массива:')
print('data   :', A_csr.data.round(3))
print('indices:', A_csr.indices, '(номера столбцов ненулевых элементов)')
print('indptr :', A_csr.indptr, '(где в data начинается каждая строка)')

Формат: csr | форма: (5, 5)
Ненулевых элементов: 5 из 25 - плотность 0.2

Плотный вид:
[[0.    0.    0.761 0.786 0.   ]
 [0.128 0.    0.    0.    0.   ]
 [0.    0.    0.    0.976 0.   ]
 [0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.45 ]]

CSR хранит три массива:
data   : [0.761 0.786 0.128 0.976 0.45 ]
indices: [2 3 0 3 4] (номера столбцов ненулевых элементов)
indptr : [0 2 3 4 4 5] (где в data начинается каждая строка)


### 4.2 Преобразование в COO и CSC

In [19]:
A_coo = A_csr.tocoo()
A_csc = A_csr.tocsc()

print('COO (список координат):')
print('  row :', A_coo.row)
print('  col :', A_coo.col)
print('  data:', A_coo.data.round(3))

print('\nCSC (сжатый по столбцам):')
print('  data   :', A_csc.data.round(3))
print('  indices:', A_csc.indices, '(номера строк)')
print('  indptr :', A_csc.indptr, '(начало каждого столбца)')

dense = A_csr.toarray()
print('\nВсе три формата задают одну матрицу:',
      np.array_equal(dense, A_coo.toarray()) and np.array_equal(dense, A_csc.toarray()))

COO (список координат):
  row : [0 0 1 2 4]
  col : [2 3 0 3 4]
  data: [0.761 0.786 0.128 0.976 0.45 ]

CSC (сжатый по столбцам):
  data   : [0.128 0.761 0.786 0.976 0.45 ]
  indices: [1 0 0 2 4] (номера строк)
  indptr : [0 1 1 2 4 5] (начало каждого столбца)

Все три формата задают одну матрицу: True


### 4.3 Умножение разреженной матрицы на плотный вектор

In [20]:
v = np.arange(1, 6, dtype=np.float64)
y = A_csr @ v
print('v        =', v)
print('A_csr @ v =', y.round(4))
print('Совпадает с плотным умножением:', np.allclose(y, A_csr.toarray() @ v))

v        = [1. 2. 3. 4. 5.]
A_csr @ v = [5.4277 0.1281 3.9025 0.     2.2519]
Совпадает с плотным умножением: True


### 4.4 Сравнение времени: разреженная и плотная матрицы 1000×1000, плотность 0.01

In [21]:
n = 1000
S = sp.random(n, n, density=0.01, format='csr', rng=0)
D = S.toarray()
rng = np.random.default_rng(1)
vec = rng.random(n)
M = rng.random((n, n))

print('nnz =', S.nnz, '| плотность =', S.nnz / n**2)
mem_sparse = S.data.nbytes + S.indices.nbytes + S.indptr.nbytes
print(f'Память: CSR = {mem_sparse / 1024:.0f} КБ, плотная = {D.nbytes / 1024:.0f} КБ (в {D.nbytes / mem_sparse:.0f} раз больше)')
print('Результаты совпадают:', np.allclose(S @ vec, D @ vec), np.allclose(S @ M, D @ M))

nnz = 10000 | плотность = 0.01
Память: CSR = 121 КБ, плотная = 7812 КБ (в 65 раз больше)
Результаты совпадают: True True


In [22]:
print('матрица x вектор, разреженная (CSR):')
%timeit S @ vec

матрица x вектор, разреженная (CSR):


8.28 μs ± 216 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [23]:
print('матрица x вектор, плотная:')
%timeit D @ vec

матрица x вектор, плотная:


245 μs ± 16.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [24]:
print('матрица x плотная матрица, разреженная (CSR):')
%timeit S @ M

матрица x плотная матрица, разреженная (CSR):


3.48 ms ± 177 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
print('матрица x плотная матрица, плотная:')
%timeit D @ M

матрица x плотная матрица, плотная:


28.4 ms ± 1.31 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**Вывод 4.4.** Для матрицы 1000×1000 с плотностью 0.01 (10 000 ненулевых элементов):

- матрица × вектор: CSR ≈ 8.3 мкс против ≈ 245 мкс у плотной (быстрее примерно в 30 раз);
- матрица × плотная матрица: CSR ≈ 3.5 мс против ≈ 28 мс (быстрее примерно в 8 раз);
- память: CSR ≈ 121 КБ против ≈ 7812 КБ (экономия в 65 раз).

Разреженная матрица обрабатывает только ненулевые элементы (1% от общего числа), а плотная умножает и нули. Выигрыш есть при малой плотности; при плотности в десятки процентов накладные расходы на индексы и нерегулярный доступ к памяти сводят его на нет. Форматы: CSR удобен для операций по строкам и умножения матрицы на вектор, CSC - для операций по столбцам, COO - для простого построения матрицы по координатам (затем её обычно переводят в CSR/CSC).

## Задача 5. Ускорение циклов с Cython

### 5.1 Функция на чистом Python

In [26]:
def sum_squares_py(lst):
    s = 0.0
    for x in lst:
        s += x * x
    return s

print(sum_squares_py([1.0, 2.0, 3.0]), '(ожидалось 14.0)')

14.0 (ожидалось 14.0)


### 5.2 Cython-версия в отдельном файле `.pyx` и её компиляция

In [27]:
%%writefile sum_squares.pyx
# cython: language_level=3, boundscheck=False, wraparound=False, cdivision=True

def sum_squares_list(list data):
    """Сумма квадратов элементов списка (типизированная переменная цикла)."""
    cdef double s = 0.0
    cdef double x
    for x in data:
        s += x * x
    return s


def sum_squares_arr(double[::1] a):
    """Сумма квадратов элементов массива через типизированное memoryview."""
    cdef Py_ssize_t i, n = a.shape[0]
    cdef double s = 0.0
    for i in range(n):
        s += a[i] * a[i]
    return s

Writing sum_squares.pyx


In [28]:
%%writefile setup.py
from setuptools import setup
from Cython.Build import cythonize

setup(ext_modules=cythonize('sum_squares.pyx'))

Writing setup.py


In [29]:
import subprocess
res = subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], capture_output=True, text=True)
print(res.stdout[-600:])
print('Код возврата:', res.returncode)
print(res.stderr[-600:] if res.returncode else '')

l -O3 -fstack-protector -fstack-clash-protection -U_FORTIFY_SOURCE -D_FORTIFY_SOURCE=2 -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer -fPIC -fPIC -I/usr/local/include/python3.13 -c sum_squares.c -o build/temp.linux-x86_64-cpython-313/sum_squares.o
cc -pthread -shared -Wl,-z,noexecstack -Wl,-z,relro -Wl,-z,now -Wl,--build-id=sha1 -Wl,--exclude-libs,ALL build/temp.linux-x86_64-cpython-313/sum_squares.o -L/usr/local/lib -o build/lib.linux-x86_64-cpython-313/sum_squares.cpython-313-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-313/sum_squares.cpython-313-x86_64-linux-gnu.so - 

Код возврата: 0



In [30]:
import sum_squares
print('Модуль загружен:', sum_squares.__file__.split('/')[-1])
print(sum_squares.sum_squares_list([1.0, 2.0, 3.0]), sum_squares.sum_squares_arr(np.array([1.0, 2.0, 3.0])), '(ожидалось 14.0)')

Модуль загружен: sum_squares.cpython-313-x86_64-linux-gnu.so
14.0 14.0 (ожидалось 14.0)


### 5.3 Сравнение времени на 10⁷ элементах

In [31]:
N = 10**7
rng = np.random.default_rng(7)
data_np = rng.random(N)
data_list = data_np.tolist()          # список Python из 10^7 float

ref = np.dot(data_np, data_np)
print('Эталон (np.dot):', ref)
print('Python :', np.isclose(sum_squares_py(data_list), ref))
print('Cython (list):', np.isclose(sum_squares.sum_squares_list(data_list), ref))
print('Cython (array):', np.isclose(sum_squares.sum_squares_arr(data_np), ref))

Эталон (np.dot): 3333134.8232450197
Python : True


Cython (list): True
Cython (array): True


In [32]:
print('Python, список:')
%timeit -n 1 -r 3 sum_squares_py(data_list)

Python, список:


201 ms ± 15.9 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [33]:
print('Cython, список:')
%timeit -n 1 -r 5 sum_squares.sum_squares_list(data_list)

Cython, список:
52.5 ms ± 1.15 ms per loop (mean ± std. dev. of 5 runs, 1 loop each)


In [34]:
print('Cython, массив NumPy (memoryview):')
%timeit -n 3 -r 5 sum_squares.sum_squares_arr(data_np)

Cython, массив NumPy (memoryview):
12.5 ms ± 259 μs per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [35]:
print('NumPy, np.dot (для сравнения):')
%timeit -n 3 -r 5 np.dot(data_np, data_np)

NumPy, np.dot (для сравнения):


3.7 ms ± 911 μs per loop (mean ± std. dev. of 5 runs, 3 loops each)


**Вывод 5.3.** Время на 10⁷ элементов (по результатам запуска выше):

| Вариант | Время | Ускорение относительно Python |
|---|---|---|
| Python, цикл по списку | ≈ 201 мс | 1× |
| Cython, цикл по списку | ≈ 52 мс | ≈ 4× |
| Cython, `double[::1]` (memoryview) | ≈ 12 мс | ≈ 16× |
| NumPy `np.dot` (для сравнения) | ≈ 3.7 мс | ≈ 54× |

Чистый Python на каждой итерации работает с объектами `float`, проверяет типы и создаёт новые объекты для промежуточных результатов. Cython переводит цикл в C: типизированные переменные `double` исключают накладные расходы интерпретатора, но при обходе списка элементы всё равно приходится «распаковывать» из объектов Python. Работа с типизированным буфером (memoryview над массивом NumPy) вообще не обращается к объектам Python, поэтому выигрыш максимален. Векторизованный `np.dot` ещё быстрее за счёт оптимизированных библиотек (BLAS, SIMD-инструкции).

## Общие выводы

1. **Числа с плавающей точкой** хранятся приближённо: ошибка ≈ 1e-8 у `float32` и ≈ 1e-17 у `float64`, машинное эпсилон равно 1.19·10⁻⁷ и 2.22·10⁻¹⁶ соответственно. Вычитание близких чисел приводит к катастрофической отмене, поэтому выражения нужно переписывать в устойчивой форме.
2. **Срезы** дают view, **fancy-индексация** - копию; для выбора «прямоугольного» подмассива по спискам индексов используется `np.ix_`. **Broadcasting** позволяет обходиться без циклов.
3. **Memory-mapped файлы** позволяют работать с массивами больше ОЗУ, читая только нужные страницы; порядок обхода должен совпадать с порядком хранения данных (кэш-линии).
4. **Разреженные матрицы** (CSR/CSC/COO) резко экономят память и время при малой плотности: в эксперименте в 65 раз по памяти и в 8-30 раз по времени.
5. **Cython** ускоряет цикл по сравнению с чистым Python в ≈4 раза при работе со списком и в ≈16 раз при работе с типизированным буфером; векторизация NumPy остаётся самой быстрой.